In [ ]:
!pip install -q pyyaml scikit-learn

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

for filename in uploaded.keys():

    size_mb = os.path.getsize(filename) / (1024 * 1024)

    print("=" * 60)
    print("Filename:", filename)
    print(f"Size: {size_mb:.2f} MB")

In [ ]:
from pathlib import Path

BASE = Path("/content/ECOBRIDGE_WORK")

RAW = BASE / "RAW"
EXTRACTED = BASE / "EXTRACTED"

RAW.mkdir(parents=True, exist_ok=True)
EXTRACTED.mkdir(parents=True, exist_ok=True)

print("Workspace:")
print(BASE)

In [ ]:
import shutil

for zip_file in Path("/content").glob("*.zip"):

    destination = RAW / zip_file.name

    shutil.copy2(
        zip_file,
        destination
    )

    print("Copied:", zip_file.name)

In [ ]:
import zipfile

for zip_file in RAW.glob("*.zip"):

    dataset_name = zip_file.stem

    output_folder = EXTRACTED / dataset_name

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    print("\n" + "=" * 80)
    print("EXTRACTING:")
    print(zip_file.name)

    with zipfile.ZipFile(
        zip_file,
        "r"
    ) as zip_ref:

        zip_ref.extractall(
            output_folder
        )

    print("DONE")

In [ ]:
for dataset_folder in EXTRACTED.iterdir():

    if not dataset_folder.is_dir():
        continue

    files = [
        p for p in dataset_folder.rglob("*")
        if p.is_file()
    ]

    print("\n" + "=" * 80)
    print("DATASET:")
    print(dataset_folder.name)
    print("FILES:", len(files))

    print("\nFirst 20 files:")

    for file in files[:20]:

        print(
            " ",
            file.relative_to(dataset_folder)
        )

In [ ]:
print("TFRECORD FILES")
print("=" * 80)

tfrecord_files = []

for dataset_folder in EXTRACTED.iterdir():

    if not dataset_folder.is_dir():
        continue

    records = list(
        dataset_folder.rglob("*.tfrecord")
    )

    # Sometimes the extension may be .record
    records += list(
        dataset_folder.rglob("*.record")
    )

    print("\nDATASET:", dataset_folder.name)

    for record in records:

        print(
            " ",
            record
        )

        tfrecord_files.append(
            record
        )

print(
    "\nTOTAL TFRECORD/RECORD FILES:",
    len(tfrecord_files)
)

In [ ]:
print("ANNOTATION / CLASS FILES")
print("=" * 80)

interesting_files = []

for dataset_folder in EXTRACTED.iterdir():

    if not dataset_folder.is_dir():
        continue

    for file in dataset_folder.rglob("*"):

        if not file.is_file():
            continue

        name = file.name.lower()

        if (
            "annotation" in name
            or "class" in name
            or "label" in name
            or name.endswith(".yaml")
            or name.endswith(".yml")
            or name.endswith(".csv")
            or name.endswith(".json")
        ):

            print(file)

            interesting_files.append(file)

In [ ]:
import tensorflow as tf

print(
    "Number of TFRecord files:",
    len(tfrecord_files)
)

In [ ]:
from pathlib import Path
import os

# Where we extracted the datasets
BASE = Path("/content/ECOBRIDGE_WORK")
EXTRACTED = BASE / "EXTRACTED"

print("EXTRACTED FOLDER EXISTS:", EXTRACTED.exists())
print("EXTRACTED PATH:", EXTRACTED)

print("\n" + "=" * 80)
print("DATASET FOLDERS")
print("=" * 80)

if EXTRACTED.exists():

    for folder in EXTRACTED.iterdir():

        if folder.is_dir():

            print(
                "\nDATASET:",
                folder.name
            )

            files = [
                p for p in folder.rglob("*")
                if p.is_file()
            ]

            print(
                "Total files:",
                len(files)
            )

else:

    print("EXTRACTED folder does not exist!")

In [ ]:
from collections import Counter

counts = Counter()

for file in EXTRACTED.rglob("*"):

    if file.is_file():

        ext = file.suffix.lower()

        if ext == "":
            ext = "[NO EXTENSION]"

        counts[ext] += 1

print("=" * 70)
print("FILE TYPES FOUND")
print("=" * 70)

for ext, count in counts.most_common():

    print(f"{ext:20s} {count}")

In [ ]:
print("=" * 70)
print("FOLDER STRUCTURE")
print("=" * 70)

for dataset_folder in EXTRACTED.iterdir():

    if not dataset_folder.is_dir():
        continue

    print("\nDATASET:", dataset_folder.name)

    for folder in dataset_folder.rglob("*"):

        if folder.is_dir():

            print(
                " ",
                folder.relative_to(dataset_folder)
            )

In [ ]:
print("=" * 70)
print("ANNOTATION / CLASS FILES")
print("=" * 70)

for file in EXTRACTED.rglob("*"):

    if not file.is_file():
        continue

    if file.suffix.lower() in [
        ".csv",
        ".yaml",
        ".yml",
        ".json",
        ".xml",
        ".txt"
    ]:

        print(file)

In [ ]:
import pandas as pd
from pathlib import Path

print("=" * 80)
print("READING ALL ANNOTATION CSV FILES")
print("=" * 80)

csv_files = list(
    EXTRACTED.rglob("_annotations.csv")
)

print(
    "Total CSV files found:",
    len(csv_files)
)

for csv_file in csv_files:

    print("\n" + "=" * 80)
    print("DATASET:", csv_file.parent.parent.name)
    print("SPLIT:", csv_file.parent.name)
    print("FILE:", csv_file)

    df = pd.read_csv(csv_file)

    print("\nRows:", len(df))

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 3 rows:")
    display(df.head(3))

In [ ]:
from collections import Counter

print("=" * 80)
print("ALL CLASSES IN ALL 3 DATASETS")
print("=" * 80)

for csv_file in csv_files:

    df = pd.read_csv(csv_file)

    print("\n" + "=" * 80)
    print(
        "DATASET:",
        csv_file.parent.parent.name
    )
    print(
        "SPLIT:",
        csv_file.parent.name
    )

    # Find the class column
    class_column = None

    for possible_column in [
        "class",
        "class_name",
        "label"
    ]:

        if possible_column in df.columns:

            class_column = possible_column
            break

    if class_column is None:

        print(
            "No class column found."
        )

        continue

    counts = df[class_column].value_counts()

    print("\nClasses:")

    for class_name, count in counts.items():

        print(
            f"{str(class_name):50s} {count}"
        )

In [ ]:
from collections import Counter

overall_classes = Counter()

for csv_file in csv_files:

    df = pd.read_csv(csv_file)

    if "class" in df.columns:

        for class_name in df["class"]:

            overall_classes[
                str(class_name).strip()
            ] += 1

print("=" * 80)
print("OVERALL CLASS COUNTS")
print("=" * 80)

for class_name, count in overall_classes.most_common():

    print(
        f"{class_name:50s} {count}"
    )

In [ ]:
print("=" * 80)
print("MOTOR / PLASTIC CANDIDATES")
print("=" * 80)

for class_name, count in overall_classes.most_common():

    name = class_name.lower()

    if (
        "motor" in name
        or "plastic" in name
        or "fan" in name
        or "hdd" in name
    ):

        print(
            f"{class_name:50s} {count}"
        )

In [ ]:
sample_csv = csv_files[0]

df = pd.read_csv(sample_csv)

print(
    "CSV:",
    sample_csv
)

print(
    "\nColumns:"
)

for column in df.columns:

    print(
        "-",
        column
    )

In [ ]:
import pandas as pd
from pathlib import Path
from collections import Counter

EXTRACTED = Path("/content/ECOBRIDGE_WORK/EXTRACTED")

print("=" * 90)
print("ECOBRIDGE DATASET CLASS AUDIT")
print("=" * 90)

all_dataset_classes = {}

for dataset_folder in sorted(EXTRACTED.iterdir()):

    if not dataset_folder.is_dir():
        continue

    csv_files = list(
        dataset_folder.rglob("_annotations.csv")
    )

    if not csv_files:
        continue

    print("\n")
    print("#" * 90)
    print("DATASET:", dataset_folder.name)
    print("#" * 90)

    class_counts = Counter()

    total_rows = 0

    for csv_file in sorted(csv_files):

        df = pd.read_csv(csv_file)

        total_rows += len(df)

        class_counts.update(
            df["class"]
            .dropna()
            .astype(str)
            .str.strip()
        )

        print(
            f"{csv_file.parent.name:10s} "
            f"annotations: {len(df)}"
        )

    all_dataset_classes[
        dataset_folder.name
    ] = class_counts

    print("\nTOTAL ANNOTATIONS:", total_rows)
    print("UNIQUE CLASSES:", len(class_counts))

    print("\nCLASS COUNTS")
    print("-" * 60)

    for class_name, count in class_counts.most_common():

        print(
            f"{class_name:<40} {count}"
        )


# ---------------------------------------------------------
# MOTOR / PLASTIC SEARCH
# ---------------------------------------------------------

print("\n\n")
print("=" * 90)
print("MOTOR / PLASTIC / RELATED CLASS SEARCH")
print("=" * 90)

keywords = [
    "motor",
    "plastic",
    "fan",
    "hdd",
    "hard disk",
    "casing",
    "housing"
]

for dataset_name, class_counts in all_dataset_classes.items():

    print("\nDATASET:", dataset_name)

    found = False

    for class_name, count in class_counts.most_common():

        name = class_name.lower()

        if any(
            keyword in name
            for keyword in keywords
        ):

            print(
                f"{class_name:<40} {count}"
            )

            found = True

    if not found:

        print(
            "No motor/plastic-related class found."
        )


# ---------------------------------------------------------
# OVERLAP CHECK
# ---------------------------------------------------------

print("\n\n")
print("=" * 90)
print("DATASET FILENAME OVERLAP CHECK")
print("=" * 90)

dataset_names = [
    "Balanced_E-Waste_Dataset_v1",
    "E-Waste_Dataset_v44"
]

filename_sets = {}

for dataset_name in dataset_names:

    folder = EXTRACTED / dataset_name

    filenames = {
        p.name
        for p in folder.rglob("*.jpg")
    }

    filename_sets[dataset_name] = filenames

    print(
        dataset_name,
        "unique JPG filenames:",
        len(filenames)
    )

if all(
    name in filename_sets
    for name in dataset_names
):

    common = (
        filename_sets[
            "Balanced_E-Waste_Dataset_v1"
        ]
        &
        filename_sets[
            "E-Waste_Dataset_v44"
        ]
    )

    print(
        "\nMatching filenames:",
        len(common)
    )

    if common:

        print("\nFirst 20 matching filenames:")

        for filename in list(
            sorted(common)
        )[:20]:

            print(filename)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
from collections import Counter

# Create extraction folder
BASE_DIR = Path("/content/ECOBRIDGE_WORK/EWASTE_VERSIONS")
BASE_DIR.mkdir(parents=True, exist_ok=True)

print("Uploaded ZIP files:")
for filename in uploaded.keys():
    print(" -", filename)

# Extract each ZIP into its own folder
for filename in uploaded.keys():
    zip_path = Path("/content") / filename

    # Create clean folder name
    folder_name = Path(filename).stem.replace(".v1i", "_v1").replace(".v8i", "_v8").replace(".v9i", "_v9")
    extract_dir = BASE_DIR / folder_name
    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nExtracting: {filename}")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

    print("Done:", extract_dir)

print("\n✅ Extraction completed!")

In [ ]:
# Find all annotation CSV files
csv_files = list(BASE_DIR.rglob("_annotations.csv"))

print("==========================================")
print("ANNOTATION FILES FOUND")
print("==========================================")

for csv_file in csv_files:
    print(csv_file)

print("\nTotal annotation files:", len(csv_files))


# Analyze every dataset version
dataset_summary = {}

for csv_file in csv_files:

    # Identify which version this belongs to
    path_lower = str(csv_file).lower()

    if "v8" in path_lower:
        version = "E-waste v8"
    elif "v9" in path_lower:
        version = "E-waste v9"
    elif "v1" in path_lower:
        version = "E-waste v1"
    else:
        version = csv_file.parent.name

    df = pd.read_csv(csv_file)

    # Clean class names
    df["class"] = df["class"].astype(str).str.strip()

    class_counts = df["class"].value_counts()

    # Number of images in this annotation file
    image_count = df["filename"].nunique()

    if version not in dataset_summary:
        dataset_summary[version] = {
            "annotations": 0,
            "images": set(),
            "classes": Counter()
        }

    dataset_summary[version]["annotations"] += len(df)
    dataset_summary[version]["images"].update(df["filename"].unique())
    dataset_summary[version]["classes"].update(class_counts.to_dict())


# Print summary
print("\n\n==========================================")
print("DATASET SUMMARY")
print("==========================================")

for version, data in dataset_summary.items():

    print(f"\n{'='*50}")
    print(version)
    print(f"{'='*50}")

    print("Unique images:", len(data["images"]))
    print("Total annotations:", data["annotations"])
    print("Unique classes:", len(data["classes"]))

    print("\nMotor / Plastic check:")

    motor_found = []
    plastic_found = []

    for class_name, count in data["classes"].items():
        if "motor" in class_name.lower():
            motor_found.append((class_name, count))

        if "plastic" in class_name.lower():
            plastic_found.append((class_name, count))

    if motor_found:
        print("✅ MOTOR:")
        for name, count in motor_found:
            print(f"   {name}: {count}")
    else:
        print("❌ No Motor class found")

    if plastic_found:
        print("✅ PLASTIC:")
        for name, count in plastic_found:
            print(f"   {name}: {count}")
    else:
        print("❌ No Plastic class found")

    print("\nAll classes:")
    for class_name, count in data["classes"].most_common():
        print(f"   {class_name}: {count}")

In [ ]:
from pathlib import Path
import pandas as pd

# Locate E-waste v1 annotation file
v1_csv = Path(
    "/content/ECOBRIDGE_WORK/EWASTE_VERSIONS/"
    "E-waste_v1.tensorflow/train/_annotations.csv"
)

df_motor = pd.read_csv(v1_csv)

# Clean class names
df_motor["class"] = df_motor["class"].astype(str).str.strip()

# Select only Motors
motor_df = df_motor[
    df_motor["class"].str.lower() == "motors"
].copy()

print("Motor annotations:", len(motor_df))
print("Unique motor images:", motor_df["filename"].nunique())

print("\nFirst 10 motor annotations:")
display(motor_df.head(10))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import random

# Find all image files inside v1
v1_base = Path(
    "/content/ECOBRIDGE_WORK/EWASTE_VERSIONS/"
    "E-waste_v1.tensorflow"
)

image_files = list(v1_base.rglob("*"))

# Create filename → path mapping
image_map = {}

for p in image_files:
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
        image_map[p.name] = p

# Pick up to 12 random motor images
motor_samples = motor_df.sample(
    min(12, len(motor_df)),
    random_state=42
)

plt.figure(figsize=(16, 12))

for i, (_, row) in enumerate(motor_samples.iterrows()):

    filename = row["filename"]

    if filename not in image_map:
        continue

    img_path = image_map[filename]

    img = Image.open(img_path).convert("RGB")

    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(filename[:25])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd
import hashlib
import json

# --------------------------------------------------
# PATHS
# --------------------------------------------------

V1_BASE = Path(
    "/content/ECOBRIDGE_WORK/EWASTE_VERSIONS/"
    "E-waste_v1.tensorflow"
)

CSV_PATH = V1_BASE / "train" / "_annotations.csv"

OUTPUT_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# READ ANNOTATIONS
# --------------------------------------------------

df = pd.read_csv(CSV_PATH)

df["class"] = df["class"].astype(str).str.strip()

motor_df = df[
    df["class"].str.lower() == "motors"
].copy()

print("Motor annotations:", len(motor_df))
print("Unique source images:", motor_df["filename"].nunique())

# --------------------------------------------------
# FIND ORIGINAL IMAGES
# --------------------------------------------------

image_map = {}

for p in V1_BASE.rglob("*"):
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
        image_map[p.name] = p

print("Images found:", len(image_map))

# --------------------------------------------------
# CROP MOTORS
# --------------------------------------------------

records = []

PADDING = 0.08

successful = 0
failed = 0

for idx, row in motor_df.iterrows():

    filename = row["filename"]

    if filename not in image_map:
        failed += 1
        continue

    image_path = image_map[filename]

    try:
        img = Image.open(image_path).convert("RGB")

        width, height = img.size

        xmin = int(row["xmin"])
        ymin = int(row["ymin"])
        xmax = int(row["xmax"])
        ymax = int(row["ymax"])

        # ------------------------------------------
        # Add 8% padding
        # ------------------------------------------

        box_width = xmax - xmin
        box_height = ymax - ymin

        pad_x = int(box_width * PADDING)
        pad_y = int(box_height * PADDING)

        xmin = max(0, xmin - pad_x)
        ymin = max(0, ymin - pad_y)
        xmax = min(width, xmax + pad_x)
        ymax = min(height, ymax + pad_y)

        # ------------------------------------------
        # Validate box
        # ------------------------------------------

        crop_width = xmax - xmin
        crop_height = ymax - ymin

        if crop_width < 40 or crop_height < 40:
            continue

        crop = img.crop(
            (xmin, ymin, xmax, ymax)
        )

        # ------------------------------------------
        # Create unique crop filename
        # ------------------------------------------

        source_id = hashlib.md5(
            filename.encode()
        ).hexdigest()[:10]

        crop_name = (
            f"motor_{source_id}_{idx}.jpg"
        )

        crop_path = OUTPUT_DIR / crop_name

        crop.save(
            crop_path,
            quality=95
        )

        records.append({
            "crop_filename": crop_name,
            "source_filename": filename,
            "source_width": width,
            "source_height": height,
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
            "class": "CAT_MOTORS"
        })

        successful += 1

    except Exception as e:
        failed += 1
        print("Error:", filename, e)


# --------------------------------------------------
# SAVE METADATA
# --------------------------------------------------

metadata_df = pd.DataFrame(records)

metadata_path = OUTPUT_DIR / "motor_metadata.csv"

metadata_df.to_csv(
    metadata_path,
    index=False
)

print("\n======================================")
print("MOTOR EXTRACTION COMPLETE")
print("======================================")

print("Successful crops:", successful)
print("Failed:", failed)
print("Metadata:", metadata_path)
print("Output folder:", OUTPUT_DIR)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import random

motor_files = list(
    OUTPUT_DIR.glob("motor_*.jpg")
)

print("Total extracted motor crops:", len(motor_files))

samples = random.sample(
    motor_files,
    min(16, len(motor_files))
)

plt.figure(figsize=(14, 12))

for i, img_path in enumerate(samples):

    img = Image.open(img_path)

    plt.subplot(4, 4, i + 1)
    plt.imshow(img)
    plt.title(img_path.name[:18])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import hashlib
import pandas as pd

MOTOR_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS"
)

motor_files = [
    p for p in MOTOR_DIR.glob("motor_*.jpg")
]

results = []

for path in motor_files:

    try:
        img = Image.open(path).convert("RGB")

        width, height = img.size
        aspect_ratio = width / height

        # File hash
        with open(path, "rb") as f:
            file_hash = hashlib.md5(
                f.read()
            ).hexdigest()

        results.append({
            "filename": path.name,
            "width": width,
            "height": height,
            "aspect_ratio": aspect_ratio,
            "file_size_kb": path.stat().st_size / 1024,
            "hash": file_hash
        })

    except Exception as e:
        print("Error:", path.name, e)

audit_df = pd.DataFrame(results)

print("========================================")
print("MOTOR DATASET TECHNICAL AUDIT")
print("========================================")

print("Total crops:", len(audit_df))

print("\nDimensions:")
print(
    audit_df[
        ["width", "height"]
    ].describe()
)

print("\nAspect ratio:")
print(
    audit_df["aspect_ratio"].describe()
)

# Duplicate files
duplicate_groups = (
    audit_df
    .groupby("hash")
    .filter(lambda x: len(x) > 1)
)

print("\nExact duplicate crops:",
      len(duplicate_groups))

if len(duplicate_groups) > 0:
    print(
        duplicate_groups[
            ["filename", "hash"]
        ].sort_values("hash")
    )

# Suspiciously small crops
small_crops = audit_df[
    (audit_df["width"] < 80) |
    (audit_df["height"] < 80)
]

print("\nVery small crops:", len(small_crops))

if len(small_crops) > 0:
    display(small_crops)

# Extreme aspect ratios
extreme_ratio = audit_df[
    (audit_df["aspect_ratio"] > 4) |
    (audit_df["aspect_ratio"] < 0.25)
]

print("\nExtreme aspect-ratio crops:",
      len(extreme_ratio))

if len(extreme_ratio) > 0:
    display(extreme_ratio)

print("\n✅ Technical audit completed.")

In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image

# Sort files consistently
motor_files = sorted(
    MOTOR_DIR.glob("motor_*.jpg")
)

# Show 30 at a time
PAGE_SIZE = 30

for page_start in range(
    0,
    len(motor_files),
    PAGE_SIZE
):

    page_files = motor_files[
        page_start:page_start + PAGE_SIZE
    ]

    rows = math.ceil(
        len(page_files) / 5
    )

    plt.figure(
        figsize=(15, rows * 3)
    )

    for i, img_path in enumerate(page_files):

        img = Image.open(img_path)

        plt.subplot(rows, 5, i + 1)
        plt.imshow(img)

        number = page_start + i + 1

        plt.title(
            f"{number}: {img_path.name[:14]}",
            fontsize=8
        )

        plt.axis("off")

    plt.tight_layout()

    print(
        f"Showing crops {page_start + 1} "
        f"to {page_start + len(page_files)}"
    )

    plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import math

MOTOR_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS"
)

motor_files = sorted(
    MOTOR_DIR.glob("motor_*.jpg")
)

print("Total motor crops:", len(motor_files))

# Create larger contact sheets
PAGE_SIZE = 20

for page_start in range(0, len(motor_files), PAGE_SIZE):

    page_files = motor_files[
        page_start:page_start + PAGE_SIZE
    ]

    rows = math.ceil(len(page_files) / 4)

    plt.figure(
        figsize=(14, rows * 4)
    )

    for i, img_path in enumerate(page_files):

        img = Image.open(img_path)

        plt.subplot(rows, 4, i + 1)
        plt.imshow(img)

        number = page_start + i + 1

        plt.title(
            f"#{number}\n{img_path.name}",
            fontsize=8
        )

        plt.axis("off")

    plt.tight_layout()

    print(
        f"\n========== CROPS "
        f"{page_start + 1}–"
        f"{page_start + len(page_files)} =========="
    )

    plt.show()

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

MOTOR_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS"
)

CLEAN_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS_CLEAN"
)

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Clearly incorrect samples identified from
# the visual contact sheets
# --------------------------------------------------

REMOVE_INDEXES = [
    # Crops 1-20
    5, 7, 10, 18,

    # Crops 21-40
    26, 27, 30, 31, 36, 37, 38, 40,

    # Crops 41-60
    42, 43, 46, 47, 48, 49, 55,

    # Crops 61-80
    62, 64, 65, 68, 73, 77,

    # Crops 81-100
    81, 85, 86, 91, 93, 95, 96, 100,

    # Crops 101-120
    102, 103, 108, 109, 111, 118,

    # Crops 121-140
    121, 126, 127, 128, 129, 133, 135, 137, 138, 140,

    # Crops 141-160
    141, 142, 147, 151, 152, 154, 155, 159,

    # Crops 161-171
    164, 165, 167, 170, 171
]

# --------------------------------------------------
# Get sorted motor files
# --------------------------------------------------

motor_files = sorted(
    MOTOR_DIR.glob("motor_*.jpg")
)

print("Original motor crops:", len(motor_files))
print("Marked for removal:", len(REMOVE_INDEXES))

# --------------------------------------------------
# Copy only approved crops
# --------------------------------------------------

kept = []
removed = []

for number, path in enumerate(motor_files, start=1):

    if number in REMOVE_INDEXES:
        removed.append(path.name)
    else:
        destination = CLEAN_DIR / path.name
        shutil.copy2(path, destination)
        kept.append(path.name)

print("\n===================================")
print("MOTOR CLEANING RESULT")
print("===================================")
print("Original:", len(motor_files))
print("Removed:", len(removed))
print("Kept:", len(kept))

print("\nClean dataset:")
print(CLEAN_DIR)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

clean_files = sorted(
    CLEAN_DIR.glob("motor_*.jpg")
)

print("Clean motor crops:", len(clean_files))

samples = random.sample(
    clean_files,
    min(30, len(clean_files))
)

plt.figure(figsize=(15, 18))

for i, img_path in enumerate(samples):

    img = Image.open(img_path)

    plt.subplot(6, 5, i + 1)
    plt.imshow(img)
    plt.title(
        img_path.name[:15],
        fontsize=7
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import math

CLEAN_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS_CLEAN"
)

clean_files = sorted(
    CLEAN_DIR.glob("motor_*.jpg")
)

print("Total clean motor crops:", len(clean_files))

PAGE_SIZE = 25

for page_start in range(0, len(clean_files), PAGE_SIZE):

    page_files = clean_files[
        page_start:page_start + PAGE_SIZE
    ]

    rows = math.ceil(len(page_files) / 5)

    plt.figure(figsize=(15, rows * 3))

    for i, img_path in enumerate(page_files):

        img = Image.open(img_path)

        number = page_start + i + 1

        plt.subplot(rows, 5, i + 1)
        plt.imshow(img)

        plt.title(
            f"#{number}\n{img_path.name[:18]}",
            fontsize=8
        )

        plt.axis("off")

    plt.tight_layout()

    print(
        f"\n========== CROPS "
        f"{page_start + 1}–"
        f"{page_start + len(page_files)} =========="
    )

    plt.show()

In [ ]:
from pathlib import Path
import shutil

CLEAN_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS_CLEAN"
)

FINAL_CLEAN_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS_FINAL"
)

FINAL_CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Clearly non-motor samples from the final visual review
BAD_NUMBERS = [
    31, 41, 62, 64, 65,
    77, 93, 100, 102, 103,
    118, 126, 127, 128,
    133, 135, 137, 140,
    141, 142, 152, 155,
    157, 159, 170, 171
]

clean_files = sorted(
    CLEAN_DIR.glob("motor_*.jpg")
)

print("Current clean crops:", len(clean_files))
print("Removing:", len(BAD_NUMBERS))

kept = 0
removed = 0

for number, path in enumerate(clean_files, start=1):

    if number in BAD_NUMBERS:
        removed += 1
        continue

    shutil.copy2(
        path,
        FINAL_CLEAN_DIR / path.name
    )

    kept += 1

print("\n================================")
print("FINAL MOTOR CLEANING")
print("================================")
print("Before:", len(clean_files))
print("Removed:", removed)
print("Final motor crops:", kept)
print("Location:", FINAL_CLEAN_DIR)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

FINAL_CLEAN_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS_FINAL"
)

final_motor_files = sorted(
    FINAL_CLEAN_DIR.glob("motor_*.jpg")
)

print(
    "Final motor crops:",
    len(final_motor_files)
)

samples = random.sample(
    final_motor_files,
    min(30, len(final_motor_files))
)

plt.figure(figsize=(15, 18))

for i, img_path in enumerate(samples):

    img = Image.open(img_path)

    plt.subplot(6, 5, i + 1)
    plt.imshow(img)

    plt.title(
        img_path.name[:15],
        fontsize=7
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import math

FINAL_CLEAN_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS_FINAL"
)

final_files = sorted(
    FINAL_CLEAN_DIR.glob("motor_*.jpg")
)

print("Current final motor crops:", len(final_files))

PAGE_SIZE = 25

for page_start in range(0, len(final_files), PAGE_SIZE):

    page_files = final_files[
        page_start:page_start + PAGE_SIZE
    ]

    rows = math.ceil(len(page_files) / 5)

    plt.figure(figsize=(15, rows * 3))

    for i, img_path in enumerate(page_files):

        number = page_start + i + 1

        img = Image.open(img_path)

        plt.subplot(rows, 5, i + 1)
        plt.imshow(img)

        plt.title(
            f"#{number}\n{img_path.name[:16]}",
            fontsize=8
        )

        plt.axis("off")

    plt.tight_layout()

    print(
        f"\n========== CURRENT CROPS "
        f"{page_start + 1}–"
        f"{page_start + len(page_files)} =========="
    )

    plt.show()

In [ ]:
from pathlib import Path
import shutil
import hashlib
import pandas as pd

CURRENT_DIR = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS_FINAL"
)

FINAL_DIR = Path(
    "/content/ECOBRIDGE_WORK/CAT_MOTORS_CLEAN"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# These numbers refer to the CURRENT 99-crop numbering
BAD_NUMBERS = [33, 41, 88]

current_files = sorted(
    CURRENT_DIR.glob("motor_*.jpg")
)

print("Current crops:", len(current_files))

# --------------------------------------------------
# Remove the 3 remaining obvious wrong crops
# --------------------------------------------------

candidate_files = []

for number, path in enumerate(current_files, start=1):

    if number in BAD_NUMBERS:
        print("Removing:", number, path.name)
        continue

    candidate_files.append(path)

print("\nAfter visual cleaning:", len(candidate_files))

# --------------------------------------------------
# Remove exact duplicate images
# --------------------------------------------------

seen_hashes = set()
duplicate_files = []

for path in candidate_files:

    with open(path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()

    if file_hash in seen_hashes:
        duplicate_files.append(path)
    else:
        seen_hashes.add(file_hash)

print("Exact duplicates found:", len(duplicate_files))

# --------------------------------------------------
# Copy final unique motors
# --------------------------------------------------

final_files = []

for path in candidate_files:

    if path in duplicate_files:
        continue

    destination = FINAL_DIR / path.name
    shutil.copy2(path, destination)

    final_files.append(path.name)

print("\n======================================")
print("FINAL MOTOR DATASET")
print("======================================")
print("Original:", len(current_files))
print("Wrong crops removed:", len(BAD_NUMBERS))
print("Exact duplicates removed:", len(duplicate_files))
print("FINAL UNIQUE MOTOR CROPS:", len(final_files))
print("Location:", FINAL_DIR)

In [ ]:
import pandas as pd
import random
import shutil
from pathlib import Path

FINAL_DIR = Path(
    "/content/ECOBRIDGE_WORK/CAT_MOTORS_CLEAN"
)

METADATA_PATH = Path(
    "/content/ECOBRIDGE_WORK/PROCESSED_MOTORS/"
    "motor_metadata.csv"
)

OUTPUT_DIR = Path(
    "/content/ECOBRIDGE_WORK/CAT_MOTORS"
)

# --------------------------------------------------
# Read original metadata
# --------------------------------------------------

metadata = pd.read_csv(METADATA_PATH)

# Only final clean crops
final_names = {
    p.name for p in FINAL_DIR.glob("motor_*.jpg")
}

metadata = metadata[
    metadata["crop_filename"].isin(final_names)
].copy()

print("Final motor crops:", len(metadata))

print(
    "Original source images:",
    metadata["source_filename"].nunique()
)

# --------------------------------------------------
# Split by ORIGINAL IMAGE
# --------------------------------------------------

source_images = sorted(
    metadata["source_filename"].unique()
)

random.seed(42)
random.shuffle(source_images)

n = len(source_images)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_sources = set(
    source_images[:train_end]
)

val_sources = set(
    source_images[train_end:val_end]
)

test_sources = set(
    source_images[val_end:]
)

print("\nSOURCE IMAGE SPLIT")
print("------------------")
print("Train:", len(train_sources))
print("Validation:", len(val_sources))
print("Test:", len(test_sources))

# --------------------------------------------------
# Create folders
# --------------------------------------------------

for split in ["train", "val", "test"]:

    (OUTPUT_DIR / split).mkdir(
        parents=True,
        exist_ok=True
    )

# --------------------------------------------------
# Copy crops
# --------------------------------------------------

for _, row in metadata.iterrows():

    source = row["source_filename"]
    crop_name = row["crop_filename"]

    if source in train_sources:
        split = "train"

    elif source in val_sources:
        split = "val"

    else:
        split = "test"

    src = FINAL_DIR / crop_name
    dst = OUTPUT_DIR / split / crop_name

    shutil.copy2(src, dst)

# --------------------------------------------------
# Final counts
# --------------------------------------------------

print("\n======================================")
print("CAT_MOTORS DATASET READY")
print("======================================")

for split in ["train", "val", "test"]:

    count = len(
        list(
            (OUTPUT_DIR / split).glob("*.jpg")
        )
    )

    print(f"{split}: {count}")

print("\nLocation:")
print(OUTPUT_DIR)

In [ ]:
from google.colab import files

uploaded_plastic = files.upload()

In [ ]:
import os
import zipfile
from pathlib import Path

# ============================================================
# 1. Check uploaded ZIP
# ============================================================

print("FILES IN /content:")
for f in os.listdir("/content"):
    print(" -", f)

# Find ZIP files
zip_files = list(Path("/content").glob("*.zip"))

print("\nZIP FILES FOUND:")
for z in zip_files:
    size_mb = z.stat().st_size / (1024 * 1024)
    print(f" - {z.name} | {size_mb:.2f} MB")

if not zip_files:
    raise FileNotFoundError("No ZIP file found in /content")

# Use the most recently modified ZIP
zip_path = max(zip_files, key=lambda x: x.stat().st_mtime)

print("\nUSING ZIP:")
print(zip_path)

In [ ]:
import shutil
from pathlib import Path

source_dir = Path("/content/ECOBRIDGE_WORK/PLASTIC_SOURCE")

# Clean old extraction if it exists
if source_dir.exists():
    shutil.rmtree(source_dir)

source_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(source_dir)

print("Extraction completed!")
print("Location:", source_dir)

In [ ]:
print("TOP-LEVEL STRUCTURE:\n")

for item in sorted(source_dir.iterdir()):
    if item.is_dir():
        print("📁", item.name)
    else:
        print("📄", item.name)

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("\nFOLDER IMAGE COUNTS:\n")

for folder in sorted(source_dir.rglob("*")):
    if folder.is_dir():
        images = [
            f for f in folder.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
        ]

        if images:
            print(f"{folder}: {len(images)} images")

In [ ]:
print("\nPOSSIBLE PLASTIC FOLDERS:\n")

for folder in sorted(source_dir.rglob("*")):
    if folder.is_dir():
        name = folder.name.lower()

        if "plastic" in name:
            images = [
                f for f in folder.rglob("*")
                if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
            ]

            print(f"📁 {folder}")
            print(f"   Images: {len(images)}")

In [ ]:
from pathlib import Path
import yaml
from collections import Counter

source_dir = Path("/content/ECOBRIDGE_WORK/PLASTIC_SOURCE")

print("=" * 70)
print("DATASET FILE STRUCTURE")
print("=" * 70)

for path in sorted(source_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(source_dir))

In [ ]:
yaml_files = list(source_dir.rglob("*.yaml")) + list(source_dir.rglob("*.yml"))

print("\nYAML FILES FOUND:")
for f in yaml_files:
    print(f)

if yaml_files:
    yaml_path = yaml_files[0]

    print("\n" + "=" * 70)
    print("DATASET YAML")
    print("=" * 70)

    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    print(data)

In [ ]:
from collections import Counter
from pathlib import Path

print("\n" + "=" * 70)
print("ANNOTATION CLASS COUNTS")
print("=" * 70)

label_files = list(source_dir.rglob("labels/*.txt"))

print("Label files found:", len(label_files))

class_counts = Counter()

for label_file in label_files:
    with open(label_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            parts = line.split()

            # YOLO format:
            # class_id x_center y_center width height
            if len(parts) >= 5:
                class_id = parts[0]
                class_counts[class_id] += 1

print("\nCLASS ID COUNTS:")
for class_id, count in sorted(class_counts.items()):
    print(f"Class {class_id}: {count}")

In [ ]:
from pathlib import Path
from PIL import Image
import shutil
import csv

# ============================================================
# SETTINGS
# ============================================================

source_dir = Path("/content/ECOBRIDGE_WORK/PLASTIC_SOURCE")
output_dir = Path("/content/ECOBRIDGE_WORK/PROCESSED_PLASTICS")

# Clean previous output
if output_dir.exists():
    shutil.rmtree(output_dir)

output_dir.mkdir(parents=True, exist_ok=True)

# Plastic_Waste = class 11
PLASTIC_CLASS_ID = "11"

# Padding around bounding box
PADDING = 0.08

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

metadata = []

total_annotations = 0
successful_crops = 0
failed_crops = 0

# ============================================================
# PROCESS TRAIN / VALID / TEST
# ============================================================

for split in ["train", "valid", "test"]:

    image_dir = source_dir / split / "images"
    label_dir = source_dir / split / "labels"

    if not image_dir.exists() or not label_dir.exists():
        print(f"Skipping {split}: images/labels folder missing")
        continue

    print(f"\nProcessing {split}...")

    for label_file in sorted(label_dir.glob("*.txt")):

        # Find matching image
        image_path = None

        for ext in IMAGE_EXTENSIONS:
            candidate = image_dir / (label_file.stem + ext)
            if candidate.exists():
                image_path = candidate
                break

        if image_path is None:
            failed_crops += 1
            continue

        try:
            img = Image.open(image_path).convert("RGB")
            img_w, img_h = img.size

            with open(label_file, "r") as f:
                lines = f.readlines()

            plastic_index = 0

            for line in lines:

                parts = line.strip().split()

                if len(parts) < 5:
                    continue

                class_id = parts[0]

                if class_id != PLASTIC_CLASS_ID:
                    continue

                total_annotations += 1

                # YOLO:
                # class x_center y_center width height
                x_center = float(parts[1])
                y_center = float(parts[2])
                box_w = float(parts[3])
                box_h = float(parts[4])

                # Convert normalized coordinates → pixels
                x1 = int((x_center - box_w / 2) * img_w)
                y1 = int((y_center - box_h / 2) * img_h)
                x2 = int((x_center + box_w / 2) * img_w)
                y2 = int((y_center + box_h / 2) * img_h)

                # Padding
                pad_x = int((x2 - x1) * PADDING)
                pad_y = int((y2 - y1) * PADDING)

                x1 = max(0, x1 - pad_x)
                y1 = max(0, y1 - pad_y)
                x2 = min(img_w, x2 + pad_x)
                y2 = min(img_h, y2 + pad_y)

                # Validate crop
                if x2 <= x1 or y2 <= y1:
                    failed_crops += 1
                    continue

                crop = img.crop((x1, y1, x2, y2))

                # Save
                crop_name = (
                    f"plastic_{split}_{label_file.stem}_{plastic_index}.jpg"
                )

                crop_path = output_dir / crop_name
                crop.save(crop_path, quality=95)

                metadata.append({
                    "crop": crop_name,
                    "split": split,
                    "source_image": image_path.name,
                    "source_label": label_file.name,
                    "original_width": img_w,
                    "original_height": img_h,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2
                })

                successful_crops += 1
                plastic_index += 1

        except Exception as e:
            failed_crops += 1
            print("Error:", image_path, e)

# ============================================================
# SAVE METADATA
# ============================================================

metadata_path = output_dir / "plastic_metadata.csv"

with open(metadata_path, "w", newline="", encoding="utf-8") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=metadata[0].keys() if metadata else []
    )

    if metadata:
        writer.writeheader()
        writer.writerows(metadata)

# ============================================================
# SUMMARY
# ============================================================

unique_sources = len(
    set(row["source_image"] for row in metadata)
)

print("\n" + "=" * 70)
print("PLASTIC EXTRACTION COMPLETE")
print("=" * 70)

print("Plastic annotations found :", total_annotations)
print("Successful crops          :", successful_crops)
print("Failed crops              :", failed_crops)
print("Unique source images      :", unique_sources)

print("\nOutput:")
print(output_dir)
print("\nMetadata:")
print(metadata_path)

In [ ]:
from pathlib import Path
from PIL import Image
import hashlib
from collections import Counter

plastic_dir = Path("/content/ECOBRIDGE_WORK/PROCESSED_PLASTICS")

images = [
    p for p in plastic_dir.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
]

print("=" * 70)
print("PLASTIC DATASET TECHNICAL AUDIT")
print("=" * 70)

print("Total crops:", len(images))

# Dimensions
dimensions = []
small_images = []

for p in images:

    try:
        with Image.open(p) as img:
            w, h = img.size
            dimensions.append((w, h))

            if w < 80 or h < 80:
                small_images.append(p.name)

    except:
        pass

print("Readable images:", len(dimensions))

if dimensions:
    widths = [x[0] for x in dimensions]
    heights = [x[1] for x in dimensions]

    print("Minimum width :", min(widths))
    print("Maximum width :", max(widths))
    print("Minimum height:", min(heights))
    print("Maximum height:", max(heights))

print("\nVery small crops:", len(small_images))

# Exact duplicate detection
hashes = {}
duplicates = []

for p in images:

    with open(p, "rb") as f:
        h = hashlib.md5(f.read()).hexdigest()

    if h in hashes:
        duplicates.append((p.name, hashes[h]))
    else:
        hashes[h] = p.name

print("Exact duplicate crops:", len(duplicates))

if duplicates:
    print("\nDuplicate examples:")
    for d in duplicates[:20]:
        print(d)

In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

plastic_dir = Path("/content/ECOBRIDGE_WORK/PROCESSED_PLASTICS")

images = sorted([
    p for p in plastic_dir.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
])

# Show first 100
sample = images[:100]

cols = 10
rows = math.ceil(len(sample) / cols)

plt.figure(figsize=(20, 2 * rows))

for i, path in enumerate(sample):

    img = Image.open(path)

    ax = plt.subplot(rows, cols, i + 1)
    ax.imshow(img)
    ax.set_title(str(i + 1), fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from datasets import load_dataset

ds = load_dataset("4w4kt/waste-dataset-image")

In [ ]:
print(ds)

In [ ]:
print(ds["train"].features)

In [ ]:
print(ds["train"][0])

In [ ]:
print(ds["train"].features["label"])

In [ ]:
class_names = ds["train"].features["label"].names

for i, name in enumerate(class_names):
    print(i, "->", name)

In [ ]:
from collections import Counter

counts = Counter(ds["train"]["label"])

for label_id, count in sorted(counts.items()):
    print(f"{label_id}: {class_names[label_id]} -> {count} images")

In [ ]:
plastic_ids = [
    i for i, name in enumerate(class_names)
    if "plastic" in name.lower()
]

print("Plastic class IDs:", plastic_ids)

for i in plastic_ids:
    print(i, "->", class_names[i])

In [ ]:
import matplotlib.pyplot as plt

plastic_id = plastic_ids[0]

plastic_indices = [
    i for i, label in enumerate(ds["train"]["label"])
    if label == plastic_id
]

print("Number of plastic images:", len(plastic_indices))

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for ax, idx in zip(axes.flatten(), plastic_indices[:15]):
    ax.imshow(ds["train"][idx]["image"])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "GIZ/e-waste-dataset-yolo-labels",
    repo_type="dataset"
)

for f in files[:100]:
    print(f)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

info = api.dataset_info("GIZ/e-waste-dataset-yolo-labels")

print(info)

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "GIZ/e-waste-dataset-yolo-labels",
    repo_type="dataset"
)

for f in files[:100]:
    print(f)

In [ ]:
files = list_repo_files(
    "GIZ/e-waste-dataset-yolo-labels",
    repo_type="dataset"
)

In [ ]:
from huggingface_hub import hf_hub_download

classes_path = hf_hub_download(
    repo_id="GIZ/e-waste-dataset-yolo-labels",
    filename="YOLO with Pictures/classes.txt",
    repo_type="dataset"
)

print("Classes file:", classes_path)

with open(classes_path, "r", encoding="utf-8") as f:
    classes = [line.strip() for line in f if line.strip()]

print("\n===== CLASSES =====")
for i, name in enumerate(classes):
    print(i, "->", name)

In [ ]:
label_files = [
    f for f in files
    if "/labels/" in f.lower() and f.lower().endswith(".txt")
]

image_files = [
    f for f in files
    if "/images/" in f.lower()
]

print("Total images:", len(image_files))
print("Total label files:", len(label_files))

print("\nFirst 10 label files:")
for f in label_files[:10]:
    print(f)

In [ ]:
from collections import Counter
import os

class_counts = Counter()

for label_file in label_files:
    with open(
        hf_hub_download(
            repo_id="GIZ/e-waste-dataset-yolo-labels",
            filename=label_file,
            repo_type="dataset"
        ),
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_id = int(parts[0])
                class_counts[class_id] += 1

print("===== GIZ ANNOTATION COUNTS =====")

for class_id, count in sorted(class_counts.items()):
    class_name = classes[class_id] if class_id < len(classes) else "UNKNOWN"
    print(f"{class_id}: {class_name:15s} -> {count}")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print("Files in /content:")
for f in os.listdir("/content"):
    print(f)

In [ ]:
import os
import zipfile

zip_path = "/content/HDPE_waste.zip"
extract_path = "/content/ECOBRIDGE_WORK/HDPE_SOURCE"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed!")
print("Location:", extract_path)

In [ ]:
import os

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

image_files = []

for root, dirs, files in os.walk("/content/ECOBRIDGE_WORK/HDPE_SOURCE"):
    for f in files:
        if f.lower().endswith(image_extensions):
            image_files.append(os.path.join(root, f))

print("Total images:", len(image_files))

print("\nSample images:")
for f in image_files[:20]:
    print(f)

In [ ]:
import os
import random
import math
from PIL import Image
import matplotlib.pyplot as plt

SOURCE_DIR = "/content/ECOBRIDGE_WORK/HDPE_SOURCE"

# Find all images
image_files = []

for root, dirs, files in os.walk(SOURCE_DIR):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
            image_files.append(os.path.join(root, f))

print("Total images found:", len(image_files))

# Select 100 random images
random.seed(42)
sample_files = random.sample(image_files, min(100, len(image_files)))

# Create contact sheet
cols = 10
rows = math.ceil(len(sample_files) / cols)

plt.figure(figsize=(20, 20))

for i, file_path in enumerate(sample_files):
    try:
        img = Image.open(file_path).convert("RGB")

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(str(i + 1), fontsize=8)

    except Exception as e:
        print("Could not read:", file_path)

plt.suptitle(
    "HDPE Dataset — 100 Random Sample Images",
    fontsize=20
)

plt.tight_layout()
plt.show()

In [ ]:
import os

print("ZIP files in /content:")

for f in os.listdir("/content"):
    if f.lower().endswith(".zip"):
        size_mb = os.path.getsize("/content/" + f) / (1024 * 1024)
        print(f"{size_mb:.2f} MB  -->  {f}")

In [ ]:
import os

print("ZIP files in /content:")

for f in os.listdir("/content"):
    if f.lower().endswith(".zip"):
        size_mb = os.path.getsize("/content/" + f) / (1024 * 1024)
        print(f"{size_mb:.2f} MB --> {f}")

In [ ]:
import os
import zipfile

ZIP_PATH = "/content/HDPE_waste.zip"
SOURCE_DIR = "/content/ECOBRIDGE_WORK/HDPE_SOURCE"

os.makedirs(SOURCE_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(SOURCE_DIR)

print("✅ Extraction completed!")
print("Location:", SOURCE_DIR)

In [ ]:
import os

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

image_files = []

for root, dirs, files in os.walk(SOURCE_DIR):
    for f in files:
        if f.lower().endswith(image_extensions):
            image_files.append(os.path.join(root, f))

print("✅ Total images found:", len(image_files))

print("\nFirst 20 images:")
for f in image_files[:20]:
    print(f)

In [ ]:
import os
import random
import math
from PIL import Image
import matplotlib.pyplot as plt

SOURCE_DIR = "/content/ECOBRIDGE_WORK/HDPE_SOURCE"

# Find all images
image_files = []

for root, dirs, files in os.walk(SOURCE_DIR):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
            image_files.append(os.path.join(root, f))

print("Total images:", len(image_files))

# Take 100 random images
random.seed(42)
sample_files = random.sample(image_files, min(100, len(image_files)))

# Contact sheet
cols = 10
rows = math.ceil(len(sample_files) / cols)

plt.figure(figsize=(20, 20))

for i, file_path in enumerate(sample_files):
    try:
        img = Image.open(file_path).convert("RGB")

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(str(i + 1), fontsize=8)

    except Exception as e:
        print("Error:", file_path, e)

plt.suptitle(
    "HDPE Dataset — 100 Random Images",
    fontsize=20
)

plt.tight_layout()
plt.show()

In [ ]:
import os
import csv

BASE = "/content/ECOBRIDGE_WORK/EWASTE_VERSIONS"

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.endswith("_annotations.csv"):
            path = os.path.join(root, f)

            with open(path, "r", encoding="utf-8-sig") as file:
                reader = csv.DictReader(file)

                for row in reader:
                    if row.get("class", "").strip().lower() == "plastic":
                        print(row["filename"])

In [ ]:
import os

SEARCH_DIR = "/content/ECOBRIDGE_WORK"

print("Searching for annotation CSV files...\n")

found = []

for root, dirs, files in os.walk(SEARCH_DIR):
    for f in files:
        if f.lower().endswith(".csv"):
            path = os.path.join(root, f)
            found.append(path)
            print(path)

print("\nTotal CSV files found:", len(found))

In [ ]:
import os

print("Folders currently inside /content/ECOBRIDGE_WORK:\n")

if os.path.exists("/content/ECOBRIDGE_WORK"):
    for root, dirs, files in os.walk("/content/ECOBRIDGE_WORK"):
        level = root.replace("/content/ECOBRIDGE_WORK", "").count(os.sep)

        if level <= 2:
            print("📁", root)
            for d in dirs[:20]:
                print("   └──", d)
else:
    print("ECOBRIDGE_WORK does not exist.")

In [ ]:
import os

print("ECOBRIDGE_WORK contents:\n")

BASE = "/content/ECOBRIDGE_WORK"

if os.path.exists(BASE):
    for item in os.listdir(BASE):
        path = os.path.join(BASE, item)

        if os.path.isdir(path):
            print("📁", item)
        else:
            size_mb = os.path.getsize(path) / (1024 * 1024)
            print(f"📄 {item} ({size_mb:.2f} MB)")
else:
    print("ECOBRIDGE_WORK does not exist.")